# NIH Chest X-ray Model Evaluation

Run the cells from the project root. Set the configuration below for a labeled evaluation, class-folder evaluation, one-image prediction, or prediction-only run. Accuracy metrics are computed only when ground-truth labels are available. The model’s positive class and preprocessing must match its training setup.

In [12]:
# 1. Imports and configuration
import os, json
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
    precision_score, recall_score, roc_auc_score, classification_report, roc_curve)

MODEL_PATH = 'baseline_model.keras'
MANIFEST_PATH = 'nih_chestxray14/cleaned/cleaned_manifest.csv'
SPLIT_PATH = 'nih_chestxray14/cleaned/split_manifest.csv'
LABELS_PATH = 'nih_chestxray14/metadata/verified_image_labels.csv'
LABEL_COLUMN = 'binary_label'
OUTPUT_PATH = 'nih_chestxray14/model_test_predictions.csv'
METRICS_PATH = 'nih_chestxray14/model_test_metrics.json'
CONFUSION_PLOT = 'nih_chestxray14/confusion_matrix_test.png'
ROC_PLOT = 'nih_chestxray14/roc_curve_test.png'
CLASSES = ['NORMAL', 'PNEUMONIA']  # index 0 = negative, index 1 = positive
PREPROCESSING = 'rgb255'  # 'rgb255' or 'grayscale01'
THRESHOLD = 0.5
BATCH_SIZE = 32
LIMIT = 0  # 0 means all test images
PREDICT_ONLY = False
IMAGE_PATH = None  # set to a filename to predict one image
TEST_DIR = None  # optionally use class subfolders named by CLASSES
assert os.path.isfile(MODEL_PATH), f'Model not found: {MODEL_PATH}'
assert PREPROCESSING in {'rgb255', 'grayscale01'}
assert 0 <= THRESHOLD <= 1 and BATCH_SIZE >= 1

## 2. Load manifests and create patient-level split if needed

In [13]:
def get_manifest(manifest_path, split_path):
    df = pd.read_csv(manifest_path)
    if os.path.exists(split_path):
        splits = pd.read_csv(split_path)
        return df.merge(splits[['image', 'split']], on='image', how='inner')
    patient_ids = np.unique(df['patient_id_from_filename'].astype(str).to_numpy())
    rng = np.random.default_rng(42)
    rng.shuffle(patient_ids)
    n_train, n_val = int(len(patient_ids)*0.70), int(len(patient_ids)*0.15)
    train_ids, val_ids = set(patient_ids[:n_train]), set(patient_ids[n_train:n_train+n_val])
    def assign(patient_id):
        if patient_id in train_ids: return 'train'
        if patient_id in val_ids: return 'validation'
        return 'test'
    df['split'] = df['patient_id_from_filename'].astype(str).map(assign)
    os.makedirs(os.path.dirname(split_path), exist_ok=True)
    df[['image', 'patient_id_from_filename', 'split']].to_csv(split_path, index=False)
    return df

def read_class_folders(test_dir, classes):
    rows = []
    extensions = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}
    for label, class_name in enumerate(classes):
        class_dir = os.path.join(test_dir, class_name)
        if not os.path.isdir(class_dir):
            raise FileNotFoundError(f'Missing class folder: {class_dir}')
        for name in sorted(os.listdir(class_dir)):
            path = os.path.join(class_dir, name)
            if os.path.isfile(path) and os.path.splitext(name)[1].lower() in extensions:
                rows.append({'image': name, 'path': path, 'label': label})
    if not rows: raise ValueError(f'No supported images found under {test_dir}')
    counts = pd.Series([r['label'] for r in rows]).value_counts().to_dict()
    if set(counts) != {0, 1}: raise ValueError('Both class folders need images')
    return pd.DataFrame(rows)

if TEST_DIR:
    test_df = read_class_folders(TEST_DIR, CLASSES)
else:
    if not os.path.isfile(MANIFEST_PATH): raise FileNotFoundError(MANIFEST_PATH)
    all_df = get_manifest(MANIFEST_PATH, SPLIT_PATH)
    test_df = all_df.copy() if PREDICT_ONLY else all_df[all_df['split'] == 'test'].copy()
    if LIMIT > 0: test_df = test_df.head(LIMIT)
test_df = test_df.reset_index(drop=True)
print('Images selected:', len(test_df))
test_df.head()

Images selected: 1363


,image,patient_id_from_filename,source_member,source_sha256,duplicate_of,source_width,source_height,source_mode,source_bytes,mean_intensity,std_intensity,min_intensity,max_intensity,prepared_width,prepared_height,prepared_mode,prepared_path,status,split
0,00008649_006.png,8649,images/00008649_006.png,89abd3f10a118c0d142f28a9362323b9371aed42a586d8...,NaN,1024,1024,L,371711,118.191829,57.743155,0,255,224,224,RGB,cleaned/images/00008649_006.png,prepared,test
1,00008742_002.png,8742,images/00008742_002.png,c3d80c94b8eba2310c6eb5a34f22f70da8aa948ce136ff...,NaN,1024,1024,L,391815,154.596117,63.442998,0,255,224,224,RGB,cleaned/images/00008742_002.png,prepared,test
2,00008822_000.png,8822,images/00008822_000.png,9b6d13098dba70aeb6a1d3c14075b15955d7fc1468defb...,NaN,1024,1024,L,358423,179.486244,67.593774,0,255,224,224,RGB,cleaned/images/00008822_000.png,prepared,test
3,00008394_002.png,8394,images/00008394_002.png,9bcae5fc68138afdb2371dd4f32bcc56251847234ea544...,NaN,1024,1024,L,350814,154.772015,50.248162,7,239,224,224,RGB,cleaned/images/00008394_002.png,prepared,test
4,00008739_009.png,8739,images/00008739_009.png,6055f271d59922b914fd08043e7f146f9cfd3830c576ba...,NaN,1024,1024,L,382767,146.700179,73.762641,0,245,224,224,RGB,cleaned/images/00008739_009.png,prepared,test


## 3. Load model and prepare image batches

In [14]:
model = tf.keras.models.load_model(MODEL_PATH, compile=False)
expected = tuple(model.input_shape[1:])
if len(expected) != 3: raise ValueError(f'Expected a single image input, got {model.input_shape}')

def resolve_image(row):
    if TEST_DIR: return row['path']
    rel = str(row['prepared_path'])
    candidates = [os.path.join(os.getcwd(), rel),
        os.path.join(os.path.dirname(MANIFEST_PATH), rel),
        os.path.join(os.path.dirname(MANIFEST_PATH), 'images', str(row['image']))]
    for candidate in candidates:
        if os.path.isfile(candidate): return candidate
    raise FileNotFoundError(f"Prepared image not found: {row['image']}")

def load_image(path):
    with Image.open(path) as image:
        if PREPROCESSING == 'grayscale01':
            image = ImageOps.pad(image.convert('L'), (expected[1], expected[0]), color=0, centering=(0.5, 0.5))
            arr = np.asarray(image, dtype=np.float32) / 255.0
            return np.repeat(arr[..., None], 3, axis=-1)
        image = image.convert('RGB')
        if image.size != (expected[1], expected[0]):
            image = image.resize((expected[1], expected[0]), Image.Resampling.LANCZOS)
        return np.asarray(image, dtype=np.float32)

if IMAGE_PATH:
    one = load_image(IMAGE_PATH)
    one_score = float(np.asarray(model.predict(one[None, ...], verbose=0)).reshape(-1)[0])
    print('Positive-class score:', one_score)
    print('Prediction:', CLASSES[int(one_score >= THRESHOLD)])

## 4. Run batch predictions

In [15]:
prediction_rows = []
if not IMAGE_PATH:
    for start in range(0, len(test_df), BATCH_SIZE):
        subset = test_df.iloc[start:start+BATCH_SIZE]
        batch = np.stack([load_image(resolve_image(row)) for _, row in subset.iterrows()])
        values = np.asarray(model.predict(batch, verbose=0)).reshape((len(subset), -1))
        for (_, row), scores in zip(subset.iterrows(), values):
            record = {'image': row['image'],
                'patient_id_from_filename': row.get('patient_id_from_filename', ''),
                'split': str(row.get('split', 'test' if not PREDICT_ONLY else 'all'))}
            for j, score in enumerate(scores):
                record['raw_model_score' if len(scores)==1 else f'output_{j}'] = float(score)
            prediction_rows.append(record)
        print(f'Predicted {min(start+len(subset), len(test_df))}/{len(test_df)}', flush=True)
    predictions_df = pd.DataFrame(prediction_rows)
    if PREDICT_ONLY or (not TEST_DIR and not os.path.isfile(LABELS_PATH)):
        os.makedirs(os.path.dirname(OUTPUT_PATH) or '.', exist_ok=True)
        predictions_df.to_csv(OUTPUT_PATH, index=False)
        print('Saved raw predictions:', OUTPUT_PATH)
        print('Accuracy not calculated: ground-truth labels are required.')
    else:
        display(predictions_df.head())

Predicted 32/1363
Predicted 64/1363
Predicted 96/1363
Predicted 128/1363
Predicted 160/1363
Predicted 192/1363
Predicted 224/1363
Predicted 256/1363
Predicted 288/1363
Predicted 320/1363
Predicted 352/1363
Predicted 384/1363
Predicted 416/1363
Predicted 448/1363
Predicted 480/1363
Predicted 512/1363
Predicted 544/1363
Predicted 576/1363
Predicted 608/1363
Predicted 640/1363
Predicted 672/1363
Predicted 704/1363
Predicted 736/1363
Predicted 768/1363
Predicted 800/1363
Predicted 832/1363
Predicted 864/1363
Predicted 896/1363
Predicted 928/1363
Predicted 960/1363
Predicted 992/1363
Predicted 1024/1363
Predicted 1056/1363
Predicted 1088/1363
Predicted 1120/1363
Predicted 1152/1363
Predicted 1184/1363
Predicted 1216/1363
Predicted 1248/1363
Predicted 1280/1363
Predicted 1312/1363
Predicted 1344/1363
Predicted 1363/1363


,image,patient_id_from_filename,split,raw_model_score
0,00008649_006.png,8649,test,0.999902
1,00008742_002.png,8742,test,0.998104
2,00008822_000.png,8822,test,1.000000
3,00008394_002.png,8394,test,0.999569
4,00008739_009.png,8739,test,1.000000


## 5. Evaluate with ground-truth labels

This section requires labels for every selected image and both classes in the evaluation set.

In [16]:
# Join labels by filename. This cell is safe to re-run: labels use a separate true_label column.
if not IMAGE_PATH and not PREDICT_ONLY and (TEST_DIR or os.path.isfile(LABELS_PATH)):
    if TEST_DIR:
        predictions_df = predictions_df.copy()
        predictions_df['true_label'] = test_df['label'].to_numpy()
    else:
        labels_df = pd.read_csv(LABELS_PATH)
        required = {'image', 'binary_label', 'eligible'}
        missing = required.difference(labels_df.columns)
        if missing:
            raise ValueError(f'Verified labels CSV missing columns: {sorted(missing)}')
        if labels_df['image'].duplicated().any():
            raise ValueError('Verified labels must have at most one row per image')

        # Only exact Pneumonia and exact No Finding rows are eligible for binary scoring.
        eligible_mask = labels_df['eligible'].astype(str).str.strip().str.lower().isin({'true', '1'})
        eligible_labels = labels_df.loc[eligible_mask, ['image', 'binary_label']].copy()
        eligible_labels = eligible_labels.rename(columns={'binary_label': 'true_label'})
        eligible_labels['true_label'] = pd.to_numeric(eligible_labels['true_label'], errors='raise').astype(int)

        # Restrict evaluation to official NIH test filenames and locally available eligible images.
        official_test_path = 'nih_chestxray14/metadata/official_test_list.txt'
        if not os.path.isfile(official_test_path):
            raise FileNotFoundError(f'Official NIH test list not found: {official_test_path}')
        with open(official_test_path, encoding='utf-8-sig') as f:
            official_test_images = {line.strip() for line in f if line.strip()}
        cohort_path = 'nih_chestxray14/metadata/verified_test_manifest.csv'
        cohort_df = pd.read_csv(cohort_path)
        required_cohort = {'image', 'eligible', 'image_available'}
        if required_cohort.difference(cohort_df.columns):
            raise ValueError(f'Verified test manifest missing columns: {sorted(required_cohort.difference(cohort_df.columns))}')
        available_mask = cohort_df['image_available'].astype(str).str.strip().str.lower().isin({'true', '1'})
        cohort_eligible = cohort_df['eligible'].astype(str).str.strip().str.lower().isin({'true', '1'})
        expected_images = set(cohort_df.loc[available_mask & cohort_eligible, 'image']) & official_test_images

        if predictions_df['image'].duplicated().any():
            raise ValueError('Predictions contain duplicate filenames')
        official_predictions = predictions_df.loc[predictions_df['image'].isin(official_test_images)].copy()
        # Remove any stale merge columns left by a previous execution of this cell.
        official_predictions = official_predictions.drop(columns=['binary_label', 'true_label'], errors='ignore')
        eval_df = official_predictions.merge(
            eligible_labels, on='image', how='inner', validate='one_to_one'
        )
        matched_images = set(eval_df['image'])
        missing_predictions = expected_images - matched_images
        unexpected_predictions = matched_images - expected_images
        print(f'Locally available eligible official-test images: {len(expected_images)}')
        print(f'Eligible official-test images with predictions: {len(matched_images)}')
        print(f'Missing eligible official-test predictions: {len(missing_predictions)}')
        print(f'Predictions outside the eligible local official-test cohort: {len(unexpected_predictions)}')
        if missing_predictions or unexpected_predictions:
            raise ValueError(
                'Evaluation stopped: predictions do not cover exactly the locally available eligible official NIH test cohort. '
                'The current saved predictions may come from a provisional split.'
            )
        predictions_df = eval_df.reset_index(drop=True)

    # Do not report NIH metrics until original training preprocessing and the validation-selected
    # threshold/class mapping have been documented and verified.
    raise RuntimeError(
        'NIH metric calculation is paused: original model training preprocessing, positive-class meaning, '
        'and the validation-selected threshold are not verified in this project.'
    )
else:
    print('Evaluation skipped; select a labeled evaluation run to calculate metrics.')


Locally available eligible official-test images: 570
Eligible official-test images with predictions: 67
Missing eligible official-test predictions: 503
Predictions outside the eligible local official-test cohort: 0


ValueError: Evaluation stopped: predictions do not cover exactly the locally available eligible official NIH test cohort. The current saved predictions may come from a provisional split.